## Table of Contents

- [Packages](#0)
- [1. Build Vocabulary](#1)

<a name='0'></a>

# Install Packages

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import math

<a name='1'></a>

# 1. Build Vocabulary

In [2]:
class Vocabulary:

    def __init__(self):
        self.word_to_idx = {
            "<PAD>": 0,
            "<START>": 1,
            "<END>": 2,
            "<UNK>": 3,
        }

        self.idx_to_word = {
            0: "<PAD>",
            1: "<START>",
            2: "<END>",
            3: "<UNK>",
        }

    def build(self, sentences):

        for sentence in sentences:

            for word in sentence.split():

                if word not in self.word_to_idx:

                    idx = len(self.word_to_idx)

                    self.word_to_idx[word] = idx
                    self.idx_to_word[idx] = word

    def encode(self, sentence):

        tokens = sentence.split()

        return [
            self.word_to_idx.get(word, self.word_to_idx["<UNK>"])
            for word in tokens
        ]

    def decode(self, indices):

        words = []

        for idx in indices:

            word = self.idx_to_word[idx]

            if word == "<END>":
                break

            if word not in ["<START>", "<PAD>"]:
                words.append(word)

        return " ".join(words)

    def __len__(self):
        return len(self.word_to_idx)

In [3]:
pairs = [
    ("i am hungry", "je suis faim"),
    ("i am happy", "je suis heureux"),
    ("i am sad", "je suis triste"),
    ("you are happy", "tu es heureux"),
    ("you are sad", "tu es triste"),
    ("he is happy", "il est heureux"),
    ("she is happy", "elle est heureuse"),
    ("i love cats", "j aime chats"),
    ("i love dogs", "j aime chiens"),
]

In [4]:
src_vocab = Vocabulary()
tgt_vocab = Vocabulary()

src_vocab.build([src for src, tgt in pairs])
tgt_vocab.build([tgt for src, tgt in pairs])

In [5]:
src_vocab.encode("i am hungry")

[4, 5, 6]

In [6]:
src_vocab.decode([4,5,6])

'i am hungry'

<a name='2'></a>
# 2. Encoder

In [7]:
class Encoder(nn.Module):

    def __init__(
        self,
        input_size,
        embedding_dim,
        hidden_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            input_size,
            embedding_dim
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim
        )

    def forward(self, x):

        # x:
        # [sequence_length, batch_size]

        embedded = self.embedding(x)

        # embedded:
        # [sequence_length, batch_size, embedding_dim]

        outputs, hidden = self.gru(embedded)

        return hidden

# 3. Decoder

In [8]:
class Decoder(nn.Module):

    def __init__(
        self,
        output_size,
        embedding_dim,
        hidden_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            output_size,
            embedding_dim
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_size
        )

    def forward(self, x, hidden):

        # x:
        # [batch_size]

        x = x.unsqueeze(0)

        # [1, batch_size]

        embedded = self.embedding(x)

        # [1, batch_size, embedding_dim]

        output, hidden = self.gru(
            embedded,
            hidden
        )

        prediction = self.fc(
            output.squeeze(0)
        )

        return prediction, hidden

# 4. Complete Seq2Seq Implementation

In [9]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder
    ):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(
        self,
        src,
        tgt,
        teacher_forcing_ratio=0.5
    ):

        batch_size = src.shape[1]

        target_length = tgt.shape[0]

        vocab_size = len(tgt_vocab)

        outputs = torch.zeros(
            target_length,
            batch_size,
            vocab_size
        )

        hidden = self.encoder(src)

        input_token = tgt[0]

        for t in range(1, target_length):

            output, hidden = self.decoder(
                input_token,
                hidden
            )

            outputs[t] = output

            best_guess = output.argmax(1)

            use_teacher_forcing = (
                random.random()
                < teacher_forcing_ratio
            )

            if use_teacher_forcing:

                input_token = tgt[t]

            else:

                input_token = best_guess

        return outputs

In [10]:
def make_tensor(sentence, vocab):

    tokens = vocab.encode(sentence)

    tokens = (
        [vocab.word_to_idx["<START>"]]
        + tokens
        + [vocab.word_to_idx["<END>"]]
    )

    return torch.tensor(tokens).unsqueeze(1)

In [11]:
x = make_tensor(
    "i am happy",
    src_vocab
)

print(x.shape)

torch.Size([5, 1])


# 5. Instantiate the Model

## 5.1 Model

In [12]:
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(tgt_vocab)

ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HIDDEN_DIM = 256

encoder = Encoder(
    INPUT_DIM,
    ENC_EMB_DIM,
    HIDDEN_DIM
)

decoder = Decoder(
    OUTPUT_DIM,
    DEC_EMB_DIM,
    HIDDEN_DIM
)

model = Seq2Seq(
    encoder,
    decoder
)

## 5.2 Loss Function

In [13]:
criterion = nn.CrossEntropyLoss(
    ignore_index=tgt_vocab.word_to_idx["<PAD>"]
)

## 5.3 Optimizer

In [14]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

## 5.4 Training Loop

In [15]:
for epoch in range(1000):

    model.train()

    total_loss = 0

    random.shuffle(pairs)

    for src_sentence, tgt_sentence in pairs:

        src = make_tensor(
            src_sentence,
            src_vocab
        )

        tgt = make_tensor(
            tgt_sentence,
            tgt_vocab
        )

        optimizer.zero_grad()

        output = model(
            src,
            tgt,
            teacher_forcing_ratio=0.5
        )

        output_dim = output.shape[-1]

        output = output[1:].reshape(
            -1,
            output_dim
        )

        tgt = tgt[1:].reshape(-1)

        loss = criterion(
            output,
            tgt
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if epoch % 100 == 0:

        print(
            f"Epoch {epoch}, "
            f"Loss: {total_loss / len(pairs):.4f}"
        )

Epoch 0, Loss: 2.8024
Epoch 100, Loss: 0.0011
Epoch 200, Loss: 0.0003
Epoch 300, Loss: 0.0001
Epoch 400, Loss: 0.0001
Epoch 500, Loss: 0.0000
Epoch 600, Loss: 0.0000
Epoch 700, Loss: 0.0000
Epoch 800, Loss: 0.0000
Epoch 900, Loss: 0.0000


# 6. Inferencing

## 6.1 Greedy Decode

In [24]:
def greedy_decode(
    model,
    sentence,
    src_vocab,
    tgt_vocab,
    max_length=20
):

    model.eval()

    src = make_tensor(
        sentence,
        src_vocab
    )

    with torch.no_grad():

        hidden = model.encoder(src)

    input_token = torch.tensor([
        tgt_vocab.word_to_idx["<START>"]
    ])

    generated = []

    for _ in range(max_length):

        with torch.no_grad():

            output, hidden = model.decoder(
                input_token,
                hidden
            )

        next_token = output.argmax(1).item()

        if next_token == tgt_vocab.word_to_idx["<END>"]:
            break

        generated.append(next_token)

        input_token = torch.tensor([
            next_token
        ])

    return tgt_vocab.decode(generated)

In [28]:
greedy_decode(
    model,
    "i am dogs",
    src_vocab,
    tgt_vocab
)

'je suis faim'

## 6.2 Beam Search

                         <START>
                  /         |        \
               je          moi       nous
              -.4          -.8       -1.3
             /   \          |
          suis  mange      aussi
         -.6    -1.4       -1.1
          |
        heureux
         -0.8

In [30]:
def beam_search(
    model,
    sentence,
    src_vocab,
    tgt_vocab,
    beam_size=3,
    max_length=20
):

    model.eval()

    src = make_tensor(
        sentence,
        src_vocab
    )

    with torch.no_grad():

        hidden = model.encoder(src)

    start_token = tgt_vocab.word_to_idx["<START>"]
    end_token = tgt_vocab.word_to_idx["<END>"]

    beams = [
        (
            [start_token],
            0.0,
            hidden
        )
    ]

    for _ in range(max_length):

        candidates = []

        for tokens, score, hidden in beams:

            last_token = tokens[-1]

            if last_token == end_token:

                candidates.append(
                    (tokens, score, hidden)
                )

                continue

            input_token = torch.tensor([
                last_token
            ])

            with torch.no_grad():

                output, next_hidden = model.decoder(
                    input_token,
                    hidden
                )

            log_probs = torch.log_softmax(
                output,
                dim=-1
            )

            top_log_probs, top_indices = torch.topk(
                log_probs,
                beam_size
            )

            for log_prob, token in zip(
                top_log_probs[0],
                top_indices[0]
            ):

                token = token.item()

                new_tokens = (
                    tokens + [token]
                )

                new_score = (
                    score + log_prob.item()
                )

                candidates.append(
                    (
                        new_tokens,
                        new_score,
                        next_hidden
                    )
                )

        candidates.sort(
            key=lambda x: x[1],
            reverse=True
        )

        beams = candidates[:beam_size]

        if all(
            tokens[-1] == end_token
            for tokens, _, _ in beams
        ):
            break

    best_tokens, best_score, _ = beams[0]

    return (
        tgt_vocab.decode(best_tokens),
        best_score
    )

In [32]:
beam_search(
    model,
    "you are dogs",
    src_vocab,
    tgt_vocab,
    beam_size=3
)

('tu es heureux', -0.46245086192151774)

In [ ]:
@torch.no_grad()
def translate(model, sentence, max_length=10):
    model.eval()

    # 1. Tokenize the source sentence.
    words = sentence.lower().split()

    src_ids = [
        src_vocab.get(word, src_vocab["<unk>"])
        for word in words
    ]

    src = torch.tensor([src_ids], dtype=torch.long)

    # 2. Encode the entire source sequence.
    _, hidden = model.encoder(src)

    # 3. Start the decoder with SOS.
    token = torch.tensor([[SOS]], dtype=torch.long)

    result = []

    # 4. Predict one output token at a time.
    for _ in range(max_length):
        logits, hidden = model.decoder(token, hidden)

        # Select the most likely next token.
        next_id = logits[0, 0].argmax().item()

        if next_id == EOS:
            break

        result.append(tgt_idx_to_word[next_id])

        # Feed the predicted token into the next step.
        token = torch.tensor([[next_id]], dtype=torch.long)

    model.train()
    return " ".join(result)


print(translate(model, "i eat rice"))
print(translate(model, "you eat bread"))